# Phase 2 — Predictive Modeling: Baseline Panel + GBM

End-to-end walkthrough of the Phase 2 modeling pipeline: label generation, feature
engineering, baseline model comparison, GBM model, and exit-gate evaluation.

**Sections**
1. [Label generation](#1-label-generation)
2. [Feature engineering](#2-feature-engineering)
3. [Model definitions](#3-model-definitions)
4. [Harness self-tests](#4-harness-self-tests)
5. [Baseline panel](#5-baseline-panel)
6. [GBM model](#6-gbm-model)
7. [Diebold-Mariano test](#7-diebold-mariano-test)
8. [Exit gates T1–6](#8-exit-gates-t1t6)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from scipy import stats
from sklearn.linear_model import Ridge

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Environment ready.')

In [ ]:
import quant.storage.catalog as catalog
from quant.config import settings
from quant.features.labels import generate_labels, LabelResult
from quant.features.engineering import build_features
from quant.backtest.walkforward import walkforward_splits
from quant.backtest.harness import run_portfolio_backtest, BacktestResult
from quant.backtest.metrics import compute_metrics
from quant.backtest.statistics import diebold_mariano
from quant.backtest.report import format_report, summary_table
from quant.models.arima_baseline import ARIMABaseline
from quant.models.buyandhold_baseline import BuyAndHoldBaseline
from quant.models.gbm import GBMModel

print('Data root :', settings.data_root)

# Walk-forward parameters — shared across all models for fair comparison
HORIZON  = 5    # forward-return horizon (bars)
TRAIN_W  = 200  # training window (bars)
TEST_W   = 50   # test window per fold (bars)
STEP_W   = 50   # step between folds (bars)
EMBARGO  = 5    # post-purge buffer (bars)

# Phase 2.5: expanded panel — 5 DJIA names + SPY for regime context
PANEL_SYMS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'V', 'SPY']

print(f'Parameters: HORIZON={HORIZON}, TRAIN_W={TRAIN_W}, TEST_W={TEST_W}, '
      f'STEP_W={STEP_W}, EMBARGO={EMBARGO}')
print(f'Panel: {PANEL_SYMS}')

---
## 1 — Label generation

`generate_labels(prices, horizon)` returns a `LabelResult(series, horizon_bars)` namedtuple.
The coupling keeps `horizon_bars` inseparable from the label series — the purge
logic in `run_backtest` uses it directly so the two can never drift apart.

In [ ]:
# Load equity prices from catalog — load all PANEL_SYMS
syms_sql = ', '.join(f"'{s}'" for s in PANEL_SYMS)
try:
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp'])
    eq = eq.set_index('timestamp')
    print(f'Loaded {len(eq):,} rows | {eq.index.min().date()} to {eq.index.max().date()}')
    print(f'Symbols: {sorted(eq["symbol"].unique())}')
    USE_REAL_DATA = True
except Exception as exc:
    print(f'Catalog unavailable ({exc}) — using synthetic data')
    np.random.seed(42)
    idx = pd.bdate_range('2021-01-01', periods=600, tz='UTC')
    rows = []
    for sym in PANEL_SYMS:
        prices_sim = 150.0 * np.cumprod(1 + np.random.normal(0.0002, 0.012, len(idx)))
        vol_sim = np.abs(np.random.normal(1e7, 2e6, len(idx)))
        for i, ts in enumerate(idx):
            c = prices_sim[i]
            rows.append({'symbol': sym, 'timestamp': ts,
                         'open': c * 0.995, 'high': c * 1.005,
                         'low': c * 0.990, 'close': c, 'adjClose': c, 'volume': vol_sim[i]})
    eq = pd.DataFrame(rows).set_index('timestamp')
    USE_REAL_DATA = False

# Build per-symbol OHLCV DataFrames
prices_by_symbol_raw = {}
for sym in PANEL_SYMS:
    sym_df = eq[eq['symbol'] == sym][['open', 'high', 'low', 'adjClose', 'volume']].copy()
    sym_df = sym_df.rename(columns={'adjClose': 'close'}).sort_index().dropna()
    prices_by_symbol_raw[sym] = sym_df

# Generate labels for each symbol
lr_by_symbol = {sym: generate_labels(prices_by_symbol_raw[sym]['close'], horizon=HORIZON)
                for sym in PANEL_SYMS}

lr_aapl = lr_by_symbol['AAPL']
print(f'\nAAPL LabelResult:')
print(f'  horizon_bars = {lr_aapl.horizon_bars}  (pass directly to label_horizon= arg)')
print(f'  series shape = {lr_aapl.series.shape}')
print(f'  NaN count    = {lr_aapl.series.isna().sum()}  (last {HORIZON} bars — expected)')
display(lr_aapl.series.dropna().describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
syms_plot = PANEL_SYMS[:3]

for ax, sym in zip(axes, syms_plot):
    s = lr_by_symbol[sym].series.dropna()
    ax.hist(s, bins=60, color='#4C72B0', alpha=0.75, edgecolor='none')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(s.mean(), color='#C44E52', linewidth=1.2, linestyle='-', label=f'mean={s.mean():.4f}')
    ax.set_title(f'{sym} — {HORIZON}-bar forward return')
    ax.set_xlabel('Forward return')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Label distributions (forward returns are stationary — I(0) series)', y=1.02)
plt.tight_layout()
plt.show()

---
## 2 — Feature engineering

`build_features()` computes 17 point-in-time features per symbol:
- **13 price features:** `ret_1d`, `ret_5d`, `ret_21d`, `vol_21d`, `vol_63d`, `mom_21d`,
  `rsi_14`, `log_volume`, `ret_252d`, `ret_126d`, `ma200_ratio`, `ma50_ratio`, `volume_ratio`
- **3 FRED macro series** attached via backward ASOF merge: `DFF`, `DGS10`, `VIXCLS`
- **1 derived feature:** `yield_curve = DGS10 − DFF` (term spread)

Phase 2.5 added 5 trend/momentum/regime price features and VIXCLS + yield_curve to address
the mean-reversion bias observed in Phase 2 (GBM OOS Sharpe = −0.833 against a bull market).

In [ ]:
features_raw = build_features(PANEL_SYMS, prices_by_symbol_raw)

feat_aapl = features_raw['AAPL']
print(f'AAPL feature matrix: {feat_aapl.shape}')
print(f'Columns: {list(feat_aapl.columns)}')
print(f'NaN rows (rolling warmup): {feat_aapl.isna().any(axis=1).sum()}')

# mom_21d is at column index 5 — used by MomentumBaseline
print('\nFeature column indices (for array-based access):')
for i, col in enumerate(feat_aapl.columns):
    tag = '  ← mom_21d used by MomentumBaseline' if col == 'mom_21d' else ''
    print(f'  [{i}] {col}{tag}')

display(feat_aapl.dropna().tail(5))

In [ ]:
# Align features, labels, prices: drop NaN rows and take common index
prices_by_symbol = {}
features_by_symbol = {}
labels_by_symbol = {}

for sym in PANEL_SYMS:
    feat = features_raw[sym]
    lab  = lr_by_symbol[sym].series
    prc  = prices_by_symbol_raw[sym]
    valid = feat.dropna().index.intersection(lab.dropna().index)
    features_by_symbol[sym] = feat.loc[valid]
    labels_by_symbol[sym]   = lab.loc[valid]
    prices_by_symbol[sym]   = prc.loc[valid]

for sym in PANEL_SYMS:
    n = len(features_by_symbol[sym])
    print(f'{sym}: {n} usable bars  '
          f'({features_by_symbol[sym].index.min().date()} to '
          f'{features_by_symbol[sym].index.max().date()})')

# Feature time-series plot (first symbol, first 8 columns)
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()
feat_plot = features_by_symbol['AAPL']
for ax, col in zip(axes, feat_plot.columns[:8]):
    ax.plot(feat_plot.index, feat_plot[col], linewidth=0.8)
    ax.set_title(col, fontsize=10)
for ax in axes[8:]:
    ax.set_visible(False)
fig.suptitle('AAPL — price-derived features (post warmup, first 8)', y=1.01)
plt.tight_layout()
plt.show()

---
## 3 — Model definitions

Six baselines covering the difficulty spectrum:

| Model | Type | Signal source |
|-------|------|---------------|
| `NaiveBaseline` | Rule | Always long (+1) |
| `BuyAndHoldBaseline` | Rule | Always long (+1) — from package |
| `MomentumBaseline` | Rule | sign(21-day return) |
| `RandomWalkBaseline` | Statistical | Training-window mean return |
| `ARIMABaseline` | Statistical | AR(1) on I(0) forward returns — from package |
| `RidgeBaseline` | ML | Ridge regression on all features |

In [ ]:
# Inline baselines --------------------------------------------------------

class NaiveBaseline:
    """Always predicts +1 (perpetually bullish)."""
    def fit(self, X, y): return self
    def predict(self, X): return np.ones(len(X))

class MomentumBaseline:
    """Predicts sign of 21-day return (feature column index 5 = mom_21d)."""
    MOM_COL = 5  # index in feature matrix: ret_1d(0)..mom_21d(5)
    def fit(self, X, y): return self
    def predict(self, X):
        col = X[:, self.MOM_COL]
        return np.where(col > 0, 1.0, np.where(col < 0, -1.0, 0.0))

class RidgeBaseline:
    """Ridge regression on all features → continuous forward-return forecast."""
    def __init__(self, alpha: float = 1.0):
        self._model = Ridge(alpha=alpha)
    def fit(self, X, y): self._model.fit(X, y); return self
    def predict(self, X): return self._model.predict(X)

class RandomWalkBaseline:
    """Predicts the training-window mean return for all test bars."""
    def __init__(self): self._mean = 0.0
    def fit(self, X, y): self._mean = float(np.mean(y)); return self
    def predict(self, X): return np.full(len(X), self._mean)

# Package baselines -------------------------------------------------------
# ARIMABaseline — AR(1) on stationary forward returns (d=0)
# BuyAndHoldBaseline — always +1; fit() is a no-op

ALL_MODELS = {
    'Naive':       NaiveBaseline(),
    'BuyAndHold':  BuyAndHoldBaseline(),
    'Momentum':    MomentumBaseline(),
    'RandomWalk':  RandomWalkBaseline(),
    'ARIMA(1,0,0)': ARIMABaseline(order=(1, 0, 0)),
    'Ridge':       RidgeBaseline(alpha=1.0),
}
print('Models defined:', list(ALL_MODELS))

In [ ]:
# Smoke test: fit on 100 samples, predict on 10 samples
np.random.seed(0)
X_toy = np.random.randn(110, len(features_by_symbol['AAPL'].columns))
y_toy = np.random.randn(110) * 0.01

header = f"{'Model':<16}  {'Output range':^20}  {'Shape':^10}  Status"
print(header)
print('-' * 65)
for name, model in ALL_MODELS.items():
    m = copy.deepcopy(model)
    try:
        m.fit(X_toy[:100], y_toy[:100])
        preds = m.predict(X_toy[100:])
        print(f'{name:<16}  [{preds.min():+.4f}, {preds.max():+.4f}]  '
              f'{str(preds.shape):^10}  OK')
    except Exception as exc:
        print(f'{name:<16}  FAILED: {exc}')

---
## 4 — Harness self-tests

Two invariants are verified before touching real data:
1. **No-skill model** → OOS Sharpe near zero (within ±1.5)
2. **Purge constraint** → purging actually removes contaminated training samples

In [ ]:
# Self-test 1: random model produces near-zero Sharpe
class _RandomModel:
    def __init__(self, seed=42): self._rng = np.random.default_rng(seed)
    def fit(self, X, y): return self
    def predict(self, X): return self._rng.choice([-1.0, 0.0, 1.0], size=len(X))

sym_st = 'AAPL'
result_rand = run_portfolio_backtest(
    model=_RandomModel(),
    features_by_symbol={sym_st: features_by_symbol[sym_st]},
    labels_by_symbol={sym_st: labels_by_symbol[sym_st]},
    prices_by_symbol={sym_st: prices_by_symbol[sym_st]},
    train_window=TRAIN_W, test_window=TEST_W, step=STEP_W,
    label_horizon=HORIZON, embargo=EMBARGO,
)
sharpe_rand = result_rand.oos_metrics['sharpe']
assert abs(sharpe_rand) < 1.5, f'Random model OOS Sharpe too large: {sharpe_rand:.3f}'
print(f'Self-test 1 PASSED — random model OOS Sharpe = {sharpe_rand:.3f}  (|x| < 1.5)')

# Self-test 2: purge actually removes samples near the test boundary
n_test = len(features_by_symbol[sym_st])
splits_with_purge = list(walkforward_splits(
    n_test, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, label_horizon=HORIZON, embargo=EMBARGO
))
splits_no_purge = list(walkforward_splits(
    n_test, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, label_horizon=0, embargo=0  # no purge
))

if splits_with_purge and splits_no_purge:
    tr_with, te = splits_with_purge[0]
    tr_no, _   = splits_no_purge[0]
    contaminated = [i for i in tr_no if i > te[0] - HORIZON - EMBARGO]
    assert len(contaminated) > 0, 'Expected contaminated samples in unpurged splits'
    assert all(i not in tr_with for i in contaminated), 'Purge failed to remove contaminated samples'
    print(f'Self-test 2 PASSED — purge removed {len(contaminated)} contaminated samples '
          f'from fold 0 training set')

print(f'\nFolds generated: {len(splits_with_purge)}')

In [ ]:
# Split visualisation
n_vis = len(features_by_symbol['AAPL'])
splits_vis = list(walkforward_splits(
    n_vis, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, label_horizon=HORIZON, embargo=EMBARGO
))

fig, ax = plt.subplots(figsize=(12, max(3, len(splits_vis) * 0.45)))
for i, (tr, te) in enumerate(splits_vis):
    ax.barh(i, len(tr), left=tr[0], height=0.6, color='#4C72B0', alpha=0.8)
    purge_start = te[0] - HORIZON - EMBARGO
    ax.barh(i, HORIZON + EMBARGO, left=purge_start, height=0.6, color='#C44E52', alpha=0.6)
    ax.barh(i, len(te), left=te[0], height=0.6, color='#55A868', alpha=0.8)

ax.set_yticks(range(len(splits_vis)))
ax.set_yticklabels([f'Fold {i}' for i in range(len(splits_vis))])
ax.set_xlabel('Sample index')
ax.set_title(f'Purged walk-forward splits (HORIZON={HORIZON}, EMBARGO={EMBARGO})')
ax.legend(handles=[
    Patch(color='#4C72B0', alpha=0.8, label='Train (post purge+embargo)'),
    Patch(color='#C44E52', alpha=0.6, label='Purge+embargo gap'),
    Patch(color='#55A868', alpha=0.8, label='Test (OOS)'),
], loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5 — Baseline panel

All six models are run through identical walk-forward parameters via
`run_portfolio_backtest()`, which applies `sign(forecast)` to continuous
predictions (Ridge, ARIMA, RandomWalk) before the trade simulator.

Panel symbols: **AAPL + MSFT** (equal-weight portfolio, pooled training).

In [ ]:
panel_features  = {s: features_by_symbol[s] for s in PANEL_SYMS}
panel_labels    = {s: labels_by_symbol[s]   for s in PANEL_SYMS}
panel_prices    = {s: prices_by_symbol[s]   for s in PANEL_SYMS}

panel_results: dict[str, BacktestResult] = {}

for name, model in ALL_MODELS.items():
    print(f'Running {name}...', end=' ', flush=True)
    try:
        result = run_portfolio_backtest(
            model=copy.deepcopy(model),
            features_by_symbol=panel_features,
            labels_by_symbol=panel_labels,
            prices_by_symbol=panel_prices,
            train_window=TRAIN_W, test_window=TEST_W, step=STEP_W,
            label_horizon=HORIZON, embargo=EMBARGO,
            initial_capital=100_000, commission_per_share=0.005, slippage_bps=5.0,
        )
        panel_results[name] = result
        sr = result.oos_metrics.get('sharpe', float('nan'))
        dd = result.oos_metrics.get('max_drawdown', float('nan'))
        print(f'Sharpe={sr:.3f}  MaxDD={dd:.2%}')
    except Exception as exc:
        print(f'FAILED — {exc}')

print(f'\nCompleted {len(panel_results)}/{len(ALL_MODELS)} models.')

In [ ]:
# Summary table
metric_keys = ['sharpe', 'sortino', 'calmar', 'max_drawdown', 'total_return', 'annualized_return']
rows = []
for name, res in panel_results.items():
    row = {'model': name}
    for k in metric_keys:
        row[k] = res.oos_metrics.get(k, float('nan'))
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('model').sort_values('sharpe', ascending=False)

def fmt_cell(v, col):
    if not np.isfinite(v): return '—'
    if col in ('max_drawdown', 'total_return', 'annualized_return'): return f'{v:.2%}'
    return f'{v:.3f}'

styled = summary_df.style.apply(
    lambda col: ['background-color: #d4edda' if v == col.max() else
                 'background-color: #f8d7da' if v == col.min() else ''
                 for v in col], axis=0
).format({col: (lambda v, c=col: fmt_cell(v, c)) for col in metric_keys})

print('OOS metrics — all models (sorted by Sharpe, green=best, red=worst per column):')
display(styled)

In [ ]:
# Equity curves
colors_map = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

for i, (name, res) in enumerate(panel_results.items()):
    ec = res.equity_curve
    if ec.empty:
        continue
    c = colors_map[i % len(colors_map)]
    axes[0].plot(ec.index, ec / 1_000, linewidth=1.2, label=name, color=c)
    peak = ec.cummax()
    dd = (ec - peak) / peak * 100
    axes[1].plot(dd.index, dd, linewidth=0.8, color=c, alpha=0.7)

axes[0].axhline(100, color='black', linewidth=0.5, linestyle='--')
axes[0].set_ylabel('Portfolio ($k)')
axes[0].set_title('OOS equity curves — all baselines (AAPL+MSFT equal-weight)')
axes[0].legend(loc='upper left', fontsize=9, ncol=2)

axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Per-fold OOS Sharpe chart
n_folds = max((len(r.fold_metrics) for r in panel_results.values()), default=0)
if n_folds > 0:
    fig, ax = plt.subplots(figsize=(max(8, n_folds * 1.2), 4))
    x = np.arange(n_folds)
    width = 0.8 / max(len(panel_results), 1)
    for i, (name, res) in enumerate(panel_results.items()):
        sharpes = [f.get('sharpe', float('nan')) for f in res.fold_metrics]
        offset = (i - len(panel_results) / 2 + 0.5) * width
        ax.bar(x + offset, sharpes, width=width * 0.9,
               label=name, color=colors_map[i % len(colors_map)], alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Fold')
    ax.set_ylabel('OOS Sharpe')
    ax.set_title('Per-fold OOS Sharpe — baseline comparison')
    ax.set_xticks(x)
    ax.set_xticklabels([f'F{i}' for i in range(n_folds)])
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout()
    plt.show()

---
## 6 — GBM model

XGBoost gradient-boosted regressor with walk-forward hyperparameter tuning.
`RandomizedSearchCV(n_iter=50)` + `TimeSeriesSplit(n_splits=3)` runs inside each
training window only — the test fold is never visible to the search.
N=50 is both the search budget and the DSR N parameter used in T4.

Sample uniqueness weighting (López de Prado) down-weights overlapping labels:
for horizon=5, adjacent samples share 4 future bars, so middle samples receive
lower effective weight in the loss function.

**Interface note:** `predict()` returns continuous floats. `run_portfolio_backtest()` applies `sign()` internally.
Do **not** use `evaluate_panel()` — that function expects discrete {−1, 0, +1} signals.

In [ ]:
gbm_model = GBMModel(label_horizon=HORIZON, n_iter=50, n_splits=3, random_state=0)

gbm_result = run_portfolio_backtest(
    gbm_model,
    panel_features,
    panel_labels,
    panel_prices,
    label_horizon=HORIZON,
    train_window=TRAIN_W,
    test_window=TEST_W,
    step=STEP_W,
    embargo=EMBARGO,
)

gbm_oos = gbm_result.oos_metrics
gbm_is  = gbm_result.is_metrics
print('GBM OOS metrics:')
print(format_report(gbm_result))

# Feature importances from last fitted fold
try:
    feature_names = list(features_by_symbol[PANEL_SYMS[0]].columns)
    importances = gbm_model.feature_importances_
    n_imp = len(importances)
    imp_s = pd.Series(importances, index=feature_names[:n_imp]).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, 4))
    imp_s.plot.barh(ax=ax, color='#4C72B0', alpha=0.8)
    ax.set_xlabel('Importance (gain)')
    ax.set_title('GBM feature importances — last fold best estimator')
    plt.tight_layout()
    plt.show()
except RuntimeError:
    print('(Importances unavailable — no walk-forward folds generated on synthetic data)')

# Compare GBM vs all baselines
gbm_sr = gbm_oos.get('sharpe', float('nan'))
print(f'\nGBM OOS Sharpe: {gbm_sr:.3f}')
print('Baseline comparison:')
for name, result in sorted(panel_results.items(),
                           key=lambda x: x[1].oos_metrics.get('sharpe', -99), reverse=True):
    sr = result.oos_metrics.get('sharpe', float('nan'))
    marker = '✓' if gbm_sr > sr else '✗'
    print(f'  {marker} {name:<20s}: {sr:.3f}')
n_beaten = sum(gbm_sr > r.oos_metrics.get('sharpe', float('nan')) for r in panel_results.values())
print(f'\nGBM beats {n_beaten}/{len(panel_results)} baselines')

---
## 7 — Diebold-Mariano test

Tests whether **GBM** produces significantly smaller forecast errors than **Ridge** —
the linear ML baseline trained on the same features. This is the T5 gate.

Implementation: walk-forward loop collects raw predictions per fold for GBM and Ridge,
then `diebold_mariano(errors_gbm, errors_ridge, h=HORIZON, alternative='less',
small_sample_correction=True)` applies the Harvey-Leybourne-Newbold (1997)
small-sample correction and uses a t(T−1) reference distribution.

In [ ]:
# Collect per-fold predictions: GBM vs Ridge (single symbol, AAPL)
sym_dm = 'AAPL'
feat_arr = features_by_symbol[sym_dm].values
lab_arr  = labels_by_symbol[sym_dm].values
n_dm = len(feat_arr)

gbm_preds_list, ridge_preds_list, actuals_list = [], [], []

splits_dm = list(walkforward_splits(
    n_dm, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, label_horizon=HORIZON, embargo=EMBARGO
))

for fold_i, (tr_idx, te_idx) in enumerate(splits_dm):
    X_tr, y_tr = feat_arr[tr_idx], lab_arr[tr_idx]
    X_te, y_te = feat_arr[te_idx], lab_arr[te_idx]

    # GBM (n_iter=5 for speed; full n_iter=50 was used in Section 6)
    gbm_dm = GBMModel(label_horizon=HORIZON, n_iter=5, n_splits=2, random_state=fold_i)
    try:
        gbm_dm.fit(X_tr, y_tr)
        gbm_preds_list.extend(gbm_dm.predict(X_te))
    except Exception:
        gbm_preds_list.extend(np.zeros(len(te_idx)))

    # Ridge
    ridge_m = Ridge(alpha=1.0)
    ridge_m.fit(X_tr, y_tr)
    ridge_preds_list.extend(ridge_m.predict(X_te))

    actuals_list.extend(y_te)

actuals      = np.array(actuals_list)
errors_gbm   = np.array(gbm_preds_list) - actuals
errors_ridge = np.array(ridge_preds_list) - actuals

print(f'Walk-forward folds: {len(splits_dm)}')
print(f'Total OOS observations: {len(actuals)}')
print(f'GBM  RMSE: {np.sqrt(np.mean(errors_gbm**2)):.6f}')
print(f'Ridge RMSE: {np.sqrt(np.mean(errors_ridge**2)):.6f}')

In [ ]:
# Diebold-Mariano test: H0: MSE_gbm >= MSE_ridge  H1: MSE_gbm < MSE_ridge
dm_result = diebold_mariano(
    errors_gbm, errors_ridge,
    h=HORIZON, alternative='less', small_sample_correction=True
)

print('Diebold-Mariano test (GBM vs Ridge, one-sided, HLN-corrected):')
print(f'  T             = {dm_result.n_obs}')
print(f'  h (horizon)   = {dm_result.h}')
print(f'  HLN corrected = {dm_result.small_sample_corrected}')
print(f'  Statistic     = {dm_result.statistic:.4f}')
print(f'  p-value       = {dm_result.p_value:.4f}')

alpha = 0.10
if dm_result.p_value < alpha:
    print(f'\n  Result: REJECT H0 at α={alpha} — GBM has significantly lower '
          f'forecast MSE than Ridge (p={dm_result.p_value:.4f})')
else:
    print(f'\n  Result: FAIL TO REJECT H0 at α={alpha} — no significant MSE '
          f'difference (p={dm_result.p_value:.4f})')

loss_diff = errors_gbm**2 - errors_ridge**2
cum_diff  = np.cumsum(loss_diff)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(range(len(cum_diff)), cum_diff, linewidth=1.2, color='#4C72B0')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].fill_between(range(len(cum_diff)), cum_diff, 0,
                     where=(cum_diff < 0), color='#55A868', alpha=0.3, label='GBM better')
axes[0].fill_between(range(len(cum_diff)), cum_diff, 0,
                     where=(cum_diff >= 0), color='#C44E52', alpha=0.3, label='Ridge better')
axes[0].set_ylabel(r'$\sum (e_{GBM}^2 - e_{Ridge}^2)$')
axes[0].set_title('Cumulative loss differential: GBM vs Ridge')
axes[0].legend(fontsize=9)
axes[1].hist(loss_diff, bins=40, color='#4C72B0', alpha=0.75, edgecolor='none')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].axvline(np.mean(loss_diff), color='#C44E52', linewidth=1.2,
               label=f'mean={np.mean(loss_diff):.2e}')
axes[1].set_xlabel(r'$e_{GBM}^2 - e_{Ridge}^2$')
axes[1].set_ylabel('Count')
axes[1].set_title('Loss differential distribution')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 8 — Exit gates T1–6

Gates are evaluated on the **GBM model** output from Section 6.
All thresholds were set before any model ran — see
`docs/concepts/evaluation-standards.md`.

| Gate | Criterion | Threshold |
|------|-----------|----------|
| T1 | GBM OOS Sharpe + bootstrapped 95% CI lower bound | SR > 0.4 AND CI₅ > 0 |
| T2 | GBM IS/OOS Sharpe ratio | ratio < 2.0 |
| T3 | GBM beats all six baselines on OOS Sharpe, net of costs | all six |
| T4 | Deflated Sharpe Ratio (N = 50 configs) | DSR > 0.5 |
| T5 | DM test p-value (GBM vs Ridge, one-sided) | p < 0.10 |
| T6 | GBM OOS max drawdown | DD > −25% |

In [ ]:
# GBM results for exit gate evaluation
oos_ret = gbm_result.equity_curve.pct_change().dropna().values
best_sr = gbm_oos.get('sharpe', float('nan'))
best_is = gbm_is

print(f'GBM OOS Sharpe = {best_sr:.3f}')
print(f'OOS return observations: {len(oos_ret)}')

gate_results = {}  # populated below

In [ ]:
# T1: OOS Sharpe > 0.4 AND bootstrapped 95% CI lower bound > 0
np.random.seed(42)
BLOCK = 21       # ~1 trading month preserves autocorrelation
N_BOOT = 2000
n_ret = len(oos_ret)

boot_sharpes = []
for _ in range(N_BOOT):
    n_blocks = max(n_ret // BLOCK + 1, 1)
    starts = np.random.randint(0, max(n_ret - BLOCK, 1), size=n_blocks)
    boot = np.concatenate([oos_ret[s: s + BLOCK] for s in starts])[:n_ret]
    std_b = np.std(boot)
    boot_sharpes.append(np.mean(boot) / std_b * np.sqrt(252) if std_b > 0 else 0.0)

ci_lo, ci_hi = np.percentile(boot_sharpes, [2.5, 97.5])
t1_pass = bool(best_sr > 0.4 and ci_lo > 0.0)
gate_results['T1'] = {
    'pass': t1_pass,
    'value': f'SR={best_sr:.3f}, CI=[{ci_lo:.3f}, {ci_hi:.3f}]',
    'threshold': 'SR>0.4 AND CI₅>0'
}
print(f'T1: OOS Sharpe={best_sr:.3f}  95% CI=[{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'    {"PASS" if t1_pass else "FAIL"}')

# Bootstrap distribution
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(boot_sharpes, bins=60, color='#4C72B0', alpha=0.75, edgecolor='none')
ax.axvline(best_sr, color='#C44E52', linewidth=1.5, label=f'OOS SR={best_sr:.3f}')
ax.axvline(ci_lo, color='gray', linewidth=1.0, linestyle='--', label=f'CI₅={ci_lo:.3f}')
ax.axvline(ci_hi, color='gray', linewidth=1.0, linestyle='--', label=f'CI₉₅={ci_hi:.3f}')
ax.axvline(0.4, color='orange', linewidth=1.0, linestyle=':', label='Threshold=0.4')
ax.set_xlabel('Bootstrap Sharpe')
ax.set_title('T1 — Block-bootstrap Sharpe CI (GBM)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# T2: IS/OOS Sharpe ratio < 2.0
is_sr = best_is.get('sharpe', float('nan'))
ratio = (is_sr / best_sr) if abs(best_sr) > 1e-6 else float('inf')
t2_pass = bool(np.isfinite(ratio) and ratio < 2.0)
gate_results['T2'] = {
    'pass': t2_pass,
    'value': f'IS SR={is_sr:.3f}, ratio={ratio:.3f}',
    'threshold': 'ratio < 2.0'
}
print(f'T2: IS Sharpe={is_sr:.3f}  OOS Sharpe={best_sr:.3f}  ratio={ratio:.3f}')
print(f'    {"PASS" if t2_pass else "FAIL"}')

# T3: GBM beats all 6 baselines on OOS Sharpe
n_baselines = len(panel_results)
n_beaten = sum(best_sr > r.oos_metrics.get('sharpe', float('nan')) for r in panel_results.values())
t3_pass = bool(n_beaten == n_baselines)
gate_results['T3'] = {
    'pass': t3_pass,
    'value': f'GBM beats {n_beaten}/{n_baselines} baselines',
    'threshold': f'all {n_baselines}'
}
print(f'\nT3: GBM beats {n_beaten}/{n_baselines} baselines')
print(f'    {"PASS" if t3_pass else "FAIL"}')
for k, v in sorted(panel_results.items(),
                   key=lambda x: x[1].oos_metrics.get('sharpe', -99), reverse=True):
    sr = v.oos_metrics.get('sharpe', float('nan'))
    marker = '✓' if best_sr > sr else '✗'
    print(f'  {marker} {k:<20s}: {sr:.3f}')

In [ ]:
# T4: Deflated Sharpe Ratio (Bailey & López de Prado 2012)
# Adjusts for N ≤ 50 hyperparameter configurations and return non-normality.
# N=50 matches the GBMModel n_iter=50 hyperparameter search budget.
N_CONFIGS = 50
T_obs = len(oos_ret)
sr_daily = best_sr / np.sqrt(252)

skew_r = stats.skew(oos_ret)
kurt_r = stats.kurtosis(oos_ret)   # excess kurtosis (Fisher)

# Standard error of Sharpe (Mertens 2002)
se_sr = np.sqrt(
    (1 + 0.5 * sr_daily**2 - skew_r * sr_daily + (kurt_r / 4) * sr_daily**2) / T_obs
)

# Expected maximum Sharpe under null for N_CONFIGS trials
if N_CONFIGS == 1:
    e_max_sr = 0.0
else:
    e_max_sr = (
        (1 - np.euler_gamma) * stats.norm.ppf(1 - 1 / N_CONFIGS) +
        np.euler_gamma * stats.norm.ppf(1 - 1 / (N_CONFIGS * np.e))
    ) * se_sr

dsr = float(stats.norm.cdf((sr_daily - e_max_sr) / se_sr)) if se_sr > 0 else 0.5
t4_pass = bool(dsr > 0.5)
gate_results['T4'] = {
    'pass': t4_pass,
    'value': f'DSR={dsr:.3f}, N_configs={N_CONFIGS}',
    'threshold': 'DSR > 0.5'
}
print(f'T4: N_configs={N_CONFIGS}, skew={skew_r:.3f}, excess_kurt={kurt_r:.3f}')
print(f'    SE(SR)={se_sr:.5f}, E[max SR]={e_max_sr:.5f}')
print(f'    DSR = {dsr:.4f}')
print(f'    {"PASS" if t4_pass else "FAIL"}')

In [ ]:
# T5: DM test p-value < 0.10 (GBM vs Ridge, from Section 7)
dm_p = dm_result.p_value
t5_pass = bool(dm_p < 0.10)
gate_results['T5'] = {
    'pass': t5_pass,
    'value': f'p={dm_p:.4f} (DM stat={dm_result.statistic:.3f})',
    'threshold': 'p < 0.10'
}
print(f'T5: DM p-value = {dm_p:.4f}  stat = {dm_result.statistic:.3f}')
print(f'    {"PASS" if t5_pass else "FAIL"}')

# T6: GBM OOS max drawdown > -25%
max_dd = gbm_oos.get('max_drawdown', float('nan'))
t6_pass = bool(max_dd > -0.25)
gate_results['T6'] = {
    'pass': t6_pass,
    'value': f'max_drawdown={max_dd:.2%}',
    'threshold': 'DD > -25%'
}
print(f'\nT6: GBM max drawdown = {max_dd:.2%}')
print(f'    {"PASS" if t6_pass else "FAIL"}')

In [ ]:
# Gate summary
gate_df = pd.DataFrame([
    {'gate': gate, 'pass': d['pass'], 'value': d['value'], 'threshold': d['threshold']}
    for gate, d in gate_results.items()
]).set_index('gate')

n_pass = gate_df['pass'].sum()
n_total = len(gate_df)

def gate_style(v):
    if isinstance(v, bool):
        return 'background-color: #d4edda; color: #155724' if v else 'background-color: #f8d7da; color: #721c24'
    return ''

# pandas >= 2.1 uses .map(); older versions use .applymap()
styler = gate_df.style
try:
    styled_gates = styler.map(gate_style, subset=['pass'])
except AttributeError:
    styled_gates = styler.applymap(gate_style, subset=['pass'])
display(styled_gates)

print(f'\n{"="*50}')
if n_pass == n_total:
    print(f'ALL {n_total} GATES PASSED — baseline panel is production-ready')
else:
    failed = gate_df[~gate_df['pass']].index.tolist()
    print(f'{n_pass}/{n_total} gates passed | FAILED: {failed}')
    print('Phase 2 GBM model required to clear remaining gates.')
print(f'{"="*50}')